# Part I. ETL Pipeline for Pre-Processing the Files

## PLEASE RUN THE FOLLOWING CODE FOR PRE-PROCESSING THE FILES

#### Import Python packages 

In [16]:
# Import Python packages 
import pandas as pd
import cassandra
import re
import os
import glob
import numpy as np
import json
import csv

#### Creating list of filepaths to process original event csv data files

In [17]:
# checking your current working directory
print(os.getcwd())

# Get your current folder and subfolder event data
filepath = os.getcwd() + '/event_data'

# Create a for loop to create a list of files and collect each filepath
for root, dirs, files in os.walk(filepath):
    
# join the file path and roots with the subdirectories using glob
    file_path_list = glob.glob(os.path.join(root,'*'))


/home/workspace


#### Processing the files to create the data file csv that will be used for Apache Casssandra tables

In [18]:
# initiating an empty list of rows that will be generated from each file
full_data_rows_list = [] 
    
# for every filepath in the file path list 
for f in file_path_list:

# reading csv file 
    with open(f, 'r', encoding = 'utf8', newline='') as csvfile: 
        # creating a csv reader object 
        csvreader = csv.reader(csvfile) 
        next(csvreader)
        
 # extracting each data row one by one and append it        
        for line in csvreader:
            full_data_rows_list.append(line) 
            
# creating a smaller event data csv file called event_datafile_full csv that will be used to insert data into the \
# Apache Cassandra tables
csv.register_dialect('myDialect', quoting=csv.QUOTE_ALL, skipinitialspace=True)

with open('event_datafile_new.csv', 'w', encoding = 'utf8', newline='') as f:
    writer = csv.writer(f, dialect='myDialect')
    writer.writerow(['artist','firstName','gender','itemInSession','lastName','length',\
                'level','location','sessionId','song','userId'])
    for row in full_data_rows_list:
        if (row[0] == ''):
            continue
        writer.writerow((row[0], row[2], row[3], row[4], row[5], row[6], row[7], row[8], row[12], row[13], row[16]))


In [19]:
# number of rows in the csv file
with open('event_datafile_new.csv', 'r', encoding = 'utf8') as f:
    print(sum(1 for line in f))

6821


# Part II. Complete the Apache Cassandra coding portion of your project. 

## Now you are ready to work with the CSV file titled <font color=red>event_datafile_new.csv</font>, located within the Workspace directory.  The event_datafile_new.csv contains the following columns: 
- artist 
- firstName of user
- gender of user
- item number in session
- last name of user
- length of the song
- level (paid or free song)
- location of the user
- sessionId
- song title
- userId

The image below is a screenshot of what the denormalized data should appear like in the <font color=red>**event_datafile_new.csv**</font> after the code above is run:<br>

<img src="images/image_event_datafile_new.jpg">

## Begin writing your Apache Cassandra code in the cells below

#### Creating a Cluster

In [20]:
from cassandra.cluster import Cluster
cluster = Cluster(['127.0.0.1'])

session = cluster.connect()

#### Create Keyspace

In [21]:
try:
    session.execute("""
    CREATE KEYSPACE IF NOT EXISTS project1b 
    WITH REPLICATION = 
    { 'class' : 'SimpleStrategy', 'replication_factor' : 1 }"""
)

except Exception as e:
    print(e)

#### Set Keyspace

In [22]:
try:
    session.set_keyspace('project1b')
except Exception as e:
    print(e)

### Now we need to create tables to run the following queries. Remember, with Apache Cassandra you model the database tables on the queries you want to run.

## Create queries to ask the following three questions of the data

### 1. Give me the artist, song title and song's length in the music app history that was heard during  sessionId = 338, and itemInSession  = 4


### 2. Give me only the following: name of artist, song (sorted by itemInSession) and user (first and last name) for userid = 10, sessionid = 182
    

### 3. Give me every user name (first and last) in my music app history who listened to the song 'All Hands Against His Own'




In [23]:
## Query 1:  Get the artist, song title and song's length in the music app history for a particular sessionid and iteminsession
##Table name : music_app_history
##column1: sessionId
##column2: itemInSession
##column3: artist
##column4: song title
##column5: song length
##PRIMARY KEY (sessionId,itemInSession)
## We will set the primary key as sessionid and iteminsession. That should keep the data unique in the database.

query = "CREATE TABLE IF NOT EXISTS music_app_history "
query = query + "(sessionid int, iteminsession int, artist text, song text, length double,PRIMARY KEY (sessionid,iteminsession) )"
try:
    session.execute(query)
except Exception as e:
    print(e)                    

In [24]:
file = 'event_datafile_new.csv'

with open(file, encoding = 'utf8') as f:
    csvreader = csv.reader(f)
    next(csvreader)
    for line in csvreader:
        query = "INSERT INTO music_app_history (sessionid, iteminsession, artist, song, length) "
        query = query + "VALUES (%s, %s, %s, %s, %s)"

        session.execute(query, (int(line[8]), int(line[3]), line[0], line[9], float(line[5])))

#### Do a SELECT to verify that the data have been inserted into each table

In [25]:
#select * from music_app_history where sessionid = 338 and iteminsession = 4;
query = "select artist, song, length from music_app_history where sessionid = 338 and iteminsession = 4"
try:
    df = pd.DataFrame(list(session.execute(query)))
except Exception as e:
    print(e)
    
print(df.to_string(index=False))

artist                             song    length
Faithless  Music Matters (Mark Knight Dub)  495.3073


### COPY AND REPEAT THE ABOVE THREE CELLS FOR EACH OF THE THREE QUESTIONS

In [26]:
## Query 2: Get the name of artist, song (sorted by itemInSession) and user (first and last name)\
## for a particular userid and sessionid
##Table name : user_app_history
##column1: userid
##column2: sessionid
##column3: iteminsession
##column4: artist
##column5: song
##column6: first name
##column7: last name
##PRIMARY KEY ((userid,sessionid),iteminsession)
##userid and sesionid as primary key and iteminsession as a clustering column to sort song by iteminsession

query = "CREATE TABLE IF NOT EXISTS user_app_history "
query = query + "(userid int, sessionid int, iteminsession int, artist text, song text, first_name text, last_name text, PRIMARY KEY ((userid,sessionid),iteminsession) )"
try:
    session.execute(query)
except Exception as e:
    print(e)    
    
##INSERT
file = 'event_datafile_new.csv'

with open(file, encoding = 'utf8') as f:
    csvreader = csv.reader(f)
    next(csvreader)
    for line in csvreader:
        query = "INSERT INTO user_app_history (userid, sessionid, iteminsession, artist, song, first_name, last_name) "
        query = query + "VALUES (%s, %s, %s, %s, %s, %s, %s)"
        session.execute(query, ( int(line[10]), int(line[8]), int(line[3]), line[0], line[9], line[1], line[4] ))

#select * from user_app_history where userid = 10 and sessionid = 182;
query = "select iteminsession, artist, song, first_name, last_name from user_app_history where userid = 10 and sessionid = 182"
try:
    df = pd.DataFrame(list(session.execute(query)))
except Exception as e:
    print(e)

print(df.to_string(index=False))                 

iteminsession             artist                                               song first_name last_name
            0   Down To The Bone                                 Keep On Keepin' On     Sylvie      Cruz
            1       Three Drives                                        Greece 2000     Sylvie      Cruz
            2  Sebastien Tellier                                          Kilometer     Sylvie      Cruz
            3      Lonnie Gordon  Catch You Baby (Steve Pitron & Max Sanna Radio...     Sylvie      Cruz


In [27]:
##Query 3: Get every user name (first and last) in my music app history who listened to the a particular song

## for song = "All Hands Against His Own"
##Table name : song_app_history
##column1: song
##column2: userid
##column3: first name
##column4: last name

##PRIMARY KEY (song,userid)
##song with userid as the primary key will give us a unique set of users who heard the song "All Hands Against His Own" atleast once

query = "CREATE TABLE IF NOT EXISTS song_app_history "
query = query + "(song text, userid int, first_name text, last_name text, PRIMARY KEY (song, userid) )"
try:
    session.execute(query)
except Exception as e:
    print(e)    
    
file = 'event_datafile_new.csv'

with open(file, encoding = 'utf8') as f:
    csvreader = csv.reader(f)
    next(csvreader) 
    for line in csvreader:
        query = "INSERT INTO song_app_history (song, userid, first_name, last_name) "
        query = query + "VALUES (%s, %s, %s, %s)"

        session.execute(query, ( line[9], int(line[10]), line[1], line[4]  ))

#select * from music_app_history_3 where song = 'All Hands Against His Own';
query = "select first_name, last_name from song_app_history where song = 'All Hands Against His Own'"
try:
    df = pd.DataFrame(list(session.execute(query)))
except Exception as e:
    print(e)
    
print(df.to_string(index=False))   

first_name last_name
Jacqueline     Lynch
     Tegan    Levine
      Sara   Johnson


### Drop the tables before closing out the sessions

In [28]:
query = "drop table music_app_history"
try:
    rows = session.execute(query)
except Exception as e:
    print(e)

query = "drop table user_app_history"
try:
    rows = session.execute(query)
except Exception as e:
    print(e)
    
query = "drop table song_app_history"
try:
    rows = session.execute(query)
except Exception as e:
    print(e)

### Close the session and cluster connection¶

In [29]:
session.shutdown()
cluster.shutdown()